# FastAPI from zero: typed endpoints, async, lifespan

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 25 §25.1–25.2 · type: walkthrough*  🔧 *This builds a real endpoint.*

**The promise:** you will stand up a validated, self-documenting `POST /ask` endpoint from Pydantic models, drive it with `TestClient` (no live server, no ports), inject a dependency with `Depends`, manage a shared resource with a `lifespan` hook, and feel the event-loop-blocking trap that tanks real FastAPI servers — then fix it.

## 🧠 Why this matters

FastAPI rewards exactly one habit: **declare your types, and the framework does the rest.** One Pydantic model gives you request validation, clean `422` errors with field-level detail, response serialization, and an OpenAPI schema — all from the same declaration, with no glue code.

Two facts make it the right backbone for an AI service. First, it is **async-native**: while one request waits on a slow model call, the same thread serves others (the concurrency model from Ch 4). Second, that async strength is also its sharpest edge — a single *blocking* call inside an `async def` handler freezes the whole event loop and stalls **every** concurrent request, not just its own. Get the types and the async discipline right and you have a backend that holds under load.

## Objectives & prerequisites

**By the end you can:**

- Define `AskRequest`/`AskResponse` Pydantic models and a `POST /ask` endpoint with a `response_model`.
- Drive it with `TestClient` — in-process ASGI, no port bound — and read the generated `/openapi.json`.
- Predict and verify the `422` a malformed body produces.
- Inject a dependency with `Depends` to keep the handler thin and testable.
- Open and close a shared resource **once per process** with a `lifespan` hook.
- Diagnose the event-loop-blocking pitfall and fix it with `run_in_threadpool`.

**Prereqs:** Ch 24 (HTTP semantics, status codes); Ch 4 (async). Ch 15 (Pydantic / structured outputs) helpful. Packages: `fastapi`, `httpx` (powers `TestClient`), `pydantic` — all in `requirements.txt`.

**Run first:** the Setup cell. Defaults to `MOCK=1`, so the agent call is stubbed and the whole notebook is **free and offline** — the server never binds a port.

In [ ]:
# --- Setup -------------------------------------------------------------------
import asyncio
import os
import random
import time

from dotenv import load_dotenv

load_dotenv()  # never hardcode keys; secrets come from the environment only

# MOCK=1 (default): run_agent returns a canned answer -> free, offline,
# deterministic. MOCK=0 would make one short live model call (see run_agent).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(25)  # determinism for anything stochastic in this notebook

MODEL = "claude-opus-4-8"  # the book's default model (only used if MOCK=0)

if MOCK:
    print("MOCK mode: agent call is stubbed. No API key needed, nothing billed.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Set it in your environment/.env, or use COMPANION_MOCK=1."
        )
    print(f"LIVE mode: run_agent will call {MODEL} once per /ask request.")

## 1 · The one habit: declare your types

Two Pydantic models describe the shapes that cross the wire. `AskRequest` is what the client must send (with a validated default for `max_steps`); `AskResponse` is what we promise to return. These are the exact shapes from the book — FastAPI will validate inbound requests against `AskRequest`, serialize outbound responses against `AskResponse`, and document both.

In [ ]:
from pydantic import BaseModel, Field


class AskRequest(BaseModel):
    question: str
    max_steps: int = Field(default=8, ge=1, le=32)  # validated, with a default


class AskResponse(BaseModel):
    answer: str
    citations: list[str] = []


# A small result type so the mock and live agent paths return the same shape.
class AgentResult:
    def __init__(self, text, sources):
        self.text = text
        self.sources = sources


async def run_agent(question: str, max_steps: int = 8) -> AgentResult:
    """The agent behind the API. MOCK returns a canned, realistic answer; the
    live branch shows the one short call you'd make with COMPANION_MOCK=0."""
    if MOCK:
        await asyncio.sleep(0)  # a real awaitable, but instant and deterministic
        return AgentResult(
            text=f"Mocked answer to: {question!r} (planned for up to {max_steps} steps).",
            sources=["https://example.com/policy", "https://example.com/faq"],
        )
    import anthropic  # imported only on the live path

    client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY from the env
    resp = await client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": question}],
    )
    return AgentResult(text=resp.content[0].text, sources=[])


print("AskRequest JSON schema (this is what feeds /docs):")
import json

print(json.dumps(AskRequest.model_json_schema(), indent=2))

## 2 · 🔧 Build `POST /ask` and drive it with `TestClient`

Here is the complete endpoint from the book: an `async def` handler that takes a validated `AskRequest`, calls the agent, and returns an `AskResponse`. Because we declare `response_model=AskResponse`, FastAPI serializes and (for free editions) filters the output to that shape.

`TestClient` runs the app **in-process over ASGI** — no `uvicorn`, no port, no `await` at the call site. It's the same client your test suite uses, which is why notebooks and CI can exercise a real FastAPI app with zero infrastructure.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Agent API (Ch 25 walkthrough)")


@app.post("/ask", response_model=AskResponse)
async def ask(req: AskRequest) -> AskResponse:
    result = await run_agent(req.question, max_steps=req.max_steps)
    return AskResponse(answer=result.text, citations=result.sources)


client = TestClient(app)  # in-process ASGI; binds no socket

r = client.post("/ask", json={"question": "What is your refund policy?"})
print("status:", r.status_code)
print("body  :", json.dumps(r.json(), indent=2))

**What you just saw.** A `200` with a body that matches `AskResponse` exactly — `answer` plus `citations`. We never started a server or awaited the handler ourselves; `TestClient` drove the ASGI app and ran the event loop for us. The same declaration also produced live API docs — let's read the machine-readable version.

In [ ]:
# FastAPI generated this from the type hints alone. The browser /docs page is just
# a renderer over this document; here we read the relevant slice of it.
spec = client.get("/openapi.json").json()
ask_op = spec["paths"]["/ask"]["post"]
print("POST /ask request body schema ref:")
print(" ", ask_op["requestBody"]["content"]["application/json"]["schema"])
print("\nResponses declared:", list(ask_op["responses"].keys()))
print("AskResponse properties:", list(spec["components"]["schemas"]["AskResponse"]["properties"]))

## 3 · 🔮 Predict: what does a malformed body return?

We declared `question: str` (required) and `max_steps` as an int in `[1, 32]`. Now we send a body that breaks **both** rules: no `question`, and `max_steps` far out of range.

**Predict before running:** what HTTP status comes back — `400`, `422`, or `500`? And how much does the response tell the client about *which* fields were wrong?

In [ ]:
bad = client.post("/ask", json={"max_steps": 999})  # missing question, out-of-range steps
print("status:", bad.status_code)
for err in bad.json()["detail"]:
    print(f"  field={err['loc']!s:<28} type={err['type']:<22} msg={err['msg']}")

**What you just saw.** A clean **`422 Unprocessable Entity`** — *before your handler ran a single line*. FastAPI validated the body against `AskRequest`, found two violations, and returned field-level detail: the missing `question` and the `max_steps` ceiling. You wrote no validation code; the type declaration *is* the contract, and the framework enforces it at the boundary. That's the payoff of "declare your types."

## 4 · Inject a dependency with `Depends`

Real handlers need shared things — settings, a database session, the current user, a service. Hardcoding them makes handlers fat and untestable. **Dependency injection** via `Depends` supplies them as parameters, so the handler stays thin and you can swap a fake in tests with `app.dependency_overrides`. Here we inject a tiny `Settings` object; full DI (and request-scoped resources) lands in Ch 28.

In [ ]:
from fastapi import Depends


class Settings:
    """A stand-in for app configuration the handler shouldn't construct itself."""

    def __init__(self, default_max_steps: int = 8):
        self.default_max_steps = default_max_steps


def get_settings() -> Settings:
    # In a real app this reads env/secrets once; here it's a fixed object.
    return Settings(default_max_steps=8)


@app.get("/config")
async def config(settings: Settings = Depends(get_settings)) -> dict:
    # The handler never builds Settings; it asks for it. That's what makes it testable.
    return {"default_max_steps": settings.default_max_steps}


print("real dependency :", client.get("/config").json())

# In tests you override the dependency instead of monkeypatching internals:
app.dependency_overrides[get_settings] = lambda: Settings(default_max_steps=2)
print("overridden (test):", client.get("/config").json())
app.dependency_overrides.clear()  # always clean up overrides

**What you just saw.** The same endpoint returned different config depending on which provider was injected — no change to the handler. That override hook is how you test an endpoint that depends on a database or an external service without touching either.

## 5 · 🔧 A `lifespan`: open shared resources once per process

A connection pool, an HTTP client, an LLM client — these are expensive to build and meant to be **shared**. You want them created once when the process starts and closed cleanly on shutdown, *not* per request. The `lifespan` async context manager is FastAPI's hook for exactly that: everything before `yield` runs at startup, everything after runs at shutdown.

We'll prove it's once-per-process by counting. With `TestClient` as a context manager, entering it triggers startup and exiting triggers shutdown.

In [ ]:
from contextlib import asynccontextmanager


class FakePool:
    """Stands in for a real async connection pool (no network, deterministic)."""

    def __init__(self):
        self.opened_at = None
        self.closed = False

    async def open(self):
        self.opened_at = "startup"
        print("  [lifespan] startup: opened the shared pool (ONCE)")

    async def close(self):
        self.closed = True
        print("  [lifespan] shutdown: closed the shared pool (ONCE)")


@asynccontextmanager
async def lifespan(app: FastAPI):
    pool = FakePool()
    await pool.open()             # startup: build the shared resource
    app.state.pool = pool         # stash it on app.state for handlers to reuse
    app.state.requests_served = 0
    yield                         # <-- app serves requests here
    await pool.close()            # graceful shutdown


app2 = FastAPI(lifespan=lifespan)


@app2.get("/ping")
async def ping(req_obj=None):
    return {"pool_opened_at": app2.state.pool.opened_at}


print("entering app (startup fires once):")
with TestClient(app2) as c:  # __enter__ runs startup, __exit__ runs shutdown
    for _ in range(3):
        c.get("/ping")       # three requests, but NO repeated 'opened the pool'
    print("  served 3 requests, all sharing the same pool object")
print("exited app (shutdown fired once)")

**What you just saw.** "opened the pool" printed **once**, three requests reused the same object, and "closed the pool" printed **once** on exit. If you'd built the pool inside the handler instead, you'd open and tear down a connection pool on every request — the kind of waste that looks fine in a demo and melts a server under real traffic.

## 6 · ⚠️ Pitfall: a blocking call inside an `async` handler

This is the **number-one FastAPI performance bug**. An `async def` handler shares one event loop with every other request. If you call *synchronous, blocking* code inside it — `time.sleep`, `requests.get`, a sync DB driver, a heavy CPU loop — you don't just slow that one request: you **freeze the entire loop**, and every concurrent request waits behind it.

We'll make it visible. Two endpoints sleep for the same duration: one with blocking `time.sleep`, one with `await asyncio.sleep`. Then we fire several requests *concurrently* and time the total. Same per-request work; very different total under concurrency.

In [ ]:
import httpx
from fastapi.concurrency import run_in_threadpool

SLEEP = 0.2  # seconds of "slow I/O" per request
N = 5        # concurrent requests

perf_app = FastAPI()


@perf_app.get("/blocking")
async def blocking():
    time.sleep(SLEEP)  # ⚠️ BLOCKS the event loop: nothing else can run meanwhile
    return {"ok": True}


@perf_app.get("/nonblocking")
async def nonblocking():
    await asyncio.sleep(SLEEP)  # yields control: the loop serves others while we wait
    return {"ok": True}


@perf_app.get("/offloaded")
async def offloaded():
    # The fix when the work MUST be blocking (sync lib, CPU): push it off the loop.
    await run_in_threadpool(time.sleep, SLEEP)
    return {"ok": True}


async def hammer(path: str) -> float:
    """Fire N concurrent requests at the ASGI app and return total wall-clock."""
    transport = httpx.ASGITransport(app=perf_app)  # in-process, no port bound
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        start = time.perf_counter()
        await asyncio.gather(*(ac.get(path) for _ in range(N)))
        return time.perf_counter() - start


blocking_total = await hammer("/blocking")
nonblocking_total = await hammer("/nonblocking")
offloaded_total = await hammer("/offloaded")

print(f"{N} concurrent requests, {SLEEP}s of work each:")
print(f"  /blocking    : {blocking_total:5.2f}s   (serialized ≈ N × {SLEEP}s — the loop was frozen)")
print(f"  /nonblocking : {nonblocking_total:5.2f}s   (overlapped ≈ 1 × {SLEEP}s — the loop stayed free)")
print(f"  /offloaded   : {offloaded_total:5.2f}s   (threadpool: blocking work, loop stays free)")

**What you just saw.** The blocking endpoint took roughly `N × SLEEP` — the requests ran one after another because the loop couldn't move while `time.sleep` held it. The async version overlapped them into roughly a single `SLEEP`. The fix when blocking is unavoidable (a sync library, a CPU-bound step) is `run_in_threadpool` — it pushes the blocking call onto a worker thread so the loop stays free. The rule: **in an `async def` handler, `await` your I/O or offload it; never call blocking code inline.**

## 🎯 Senior lens: `BackgroundTasks` vs a real queue

Once an endpoint validates and dispatches in milliseconds, the question is *where the slow work goes*. FastAPI's `BackgroundTasks` runs work **after the response, in the same process** — perfect for fire-and-forget that's cheap and that you can afford to lose: writing an audit log line, sending a non-critical notification. But it is **not durable**: if the process crashes or restarts mid-task, the work is gone, and it competes for the same event loop and memory as live requests.

For anything heavy, long-running, or that *must survive a crash* — the agent run itself — you enqueue a **Celery** task instead (Ch 31). It runs in a separate worker pool, retries on failure, and survives an API restart. The senior instinct: `BackgroundTasks` for trivial after-the-response work; a durable queue the moment correctness depends on the work actually completing. Choosing `BackgroundTasks` for the agent run is how you build a backend that quietly drops jobs under load.

## Recap

- **Declare your types.** One Pydantic model gives you validation, a clean `422` with field-level detail, serialization, and OpenAPI docs — no glue code.
- **`TestClient` runs the app in-process over ASGI** — no port, no `uvicorn` — so notebooks and CI exercise a real app for free.
- **`Depends` keeps handlers thin and testable**; `app.dependency_overrides` swaps a fake in tests (full DI in Ch 28).
- **`lifespan` opens/closes shared resources once per process**, not per request — stash them on `app.state`.
- **The #1 perf bug:** a blocking call inside an `async def` freezes the whole event loop. `await` your I/O, or offload unavoidable blocking work with `run_in_threadpool`.
- **`BackgroundTasks` is in-process and non-durable**; use a real queue (Ch 31) for work that must survive a crash.

## Exercises

Each one *changes something* and asks you to predict first. (Solutions land in `solutions/` in Phase 2.)

1. **Tighten the contract.** Add a `temperature: float = Field(ge=0.0, le=1.0)` field to `AskRequest`. Predict the exact `422` detail when a client sends `temperature=2.0`, then send it through `TestClient` and confirm the `type` and `loc`.
2. **Add a streaming-free health probe.** Add `GET /healthz` returning `{"status": "ok"}` with status `200`, and assert it in a `TestClient` call. Predict whether it needs to be `async def` to work (it doesn't — explain why FastAPI accepts both).
3. **Catch a real blocking trap.** Replace the `time.sleep` in `/offloaded` with a CPU-bound loop (e.g. summing a few million ints). Predict whether `run_in_threadpool` still keeps the loop free for CPU-bound work the way it does for `time.sleep`, then measure with `hammer` and explain the GIL caveat.

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here


## Next

You have a typed, async endpoint with a proper resource lifecycle. Next: make it feel *alive* by streaming.

- ▶️ **Next notebook:** [`25-02-streaming-sse-and-websockets.ipynb`](./25-02-streaming-sse-and-websockets.ipynb) — stream tokens and structured agent events over SSE, then go two-way with WebSockets.
- 🔧 **Template (the production version of this):** [`../../../templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/) — typed routes, lifespan, and health probes as a real scaffold. You built the toy; that's the real one.
- 🎓 **Capstone:** this becomes `capstone/app/` (the API surface from §25.5); checkpoint `checkpoints/ch25-backend-api`.
- 📖 **Book:** §25.1 (FastAPI from zero) and §25.2 (async, background tasks, lifespans); DI in §28, Celery in §31.